In [1]:
# ==============================================================================
# PROYECTO BIG DATA - GRUPO 2 (REAL ESTATE)
# Módulo de Extracción Profunda Distribuida: Portal Inmobiliario
# ==============================================================================

import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- CONFIGURACIÓN PARA ASIGNACIÓN DE PÁGINAS ---
PAGINA_INICIO = 1 
PAGINA_FIN = 2
# ----------------------------------------------

os.environ["DISPLAY"] = ":99"  
os.system("pkill -9 chrome && pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.* && rm -rf /tmp/.org.chromium.Chromium.*")
print(f"🧹 Entorno virtual listo. Asignación: Páginas {PAGINA_INICIO} a la {PAGINA_FIN}.")

options = Options()
# options.add_argument("--headless=new") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0")

catalogo_urls = []
datos_finales = []

try:
    driver = webdriver.Chrome(options=options)
    
    # ==============================================================================
    # FASE 1: RECOLECCIÓN EN EL CATÁLOGO (CON PAGINACIÓN DINÁMICA)
    # ==============================================================================
    url_base = "https://www.portalinmobiliario.com/arriendo/departamento/la-serena-coquimbo"

    # El ciclo ahora respeta los límites que definieron arriba
    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):
        print(f"\n--- [FASE 1] Recolectando enlaces de la Página {pagina_actual} ---")
        
        # Cálculo de salto de página de Portal Inmobiliario
        if pagina_actual == 1:
            url_pagina = url_base
        else:
            desde = ((pagina_actual - 1) * 48) + 1
            url_pagina = f"{url_base}_Desde_{desde}_NoIndex_True"
            
        driver.get(url_pagina)

        WebDriverWait(driver, 20).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item")))
        
        for _ in range(5):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2) 

        bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")
        
        for bloque in bloques:
            try:
                # 1. URL y Título
                url = bloque.find_element(By.TAG_NAME, "a").get_attribute("href").split("#")[0].split("?")[0]
                titulo = bloque.find_element(By.CSS_SELECTOR, ".poly-component__title").get_attribute("textContent").strip()
                
                # 2. Extracción y Normalización de Ubicación (Prioridad Coquimbo)
                ubicacion_cruda = bloque.find_element(By.CSS_SELECTOR, ".poly-component__location").get_attribute("textContent").strip()
                ubi_minuscula = ubicacion_cruda.lower()
                
                if "la serena" in ubi_minuscula:
                    ubicacion = "La Serena"      
                elif "coquimbo" in ubi_minuscula:
                    ubicacion = "Coquimbo"     
                else:
                    continue 

                # 3. Extracción de Precio
                try:
                    p_text = bloque.find_element(By.CSS_SELECTOR, ".poly-price__current").get_attribute("textContent")
                except:
                    p_text = bloque.find_element(By.CSS_SELECTOR, ".poly-component__price").get_attribute("textContent")
                
                v_limpio = p_text.replace("$", "").replace(".", "").replace(",", "").replace("\n", "").strip()
                precio = float(v_limpio) if v_limpio.isdigit() else 0.0

                if url != "Sin URL" and precio > 0:
                    catalogo_urls.append({
                        "url": url,
                        "titulo": titulo,
                        "ubicacion": ubicacion,
                        "precio": precio
                    })
            except:
                continue

    print(f"Catálogo listo: {len(catalogo_urls)} propiedades encontradas. Recolectando amenidades...")

    # ==============================================================================
    # FASE 2: INSPECCIÓN PROFUNDA POR PROPIEDAD
    # ==============================================================================
    for i, propiedad in enumerate(catalogo_urls):
        print(f"[{i+1}/{len(catalogo_urls)}] Inspeccionando: {propiedad['titulo'][:30]}...")
        
        registro = {
            "responsable": "Millaray Zalazar", # Aquí cambiar el nombre de cada uno
            "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
            "titulo": propiedad["titulo"],
            "ubicacion": propiedad["ubicacion"],
            "m2": 0,
            "precio": propiedad["precio"],
            "dormitorios": 0,
            "baños": 0,
            "estacionamiento": 0,
            "piscina": 0,
            "quincho": 0,
            "terraza": 0,
            "gimnasio": 0,
            "lavanderia": 0,
            "url": propiedad["url"] 
        }

        try:
            driver.get(propiedad["url"])
            
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ".ui-pdp-collapsable__container"))
            )
            
            filas_tabla = driver.find_elements(By.CSS_SELECTOR, ".andes-table__row")
            
            for fila in filas_tabla:
                texto_fila = fila.get_attribute("textContent").lower()
                
                # Extracción Numérica
                if "superficie total" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums: registro["m2"] = int(nums[0])
                        
                elif "dormitorios" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums: registro["dormitorios"] = int(nums[0])
                        
                elif "baños" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums: registro["baños"] = int(nums[0])
                        
                elif "estacionamiento" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums and int(nums[0]) > 0:
                        registro["estacionamiento"] = int(nums[0])
                    elif "sí" in texto_fila or "si" in texto_fila:
                        registro["estacionamiento"] = 1
                
                # Extracción Booleana (1 o 0)
                elif "piscina" in texto_fila and "sí" in texto_fila:
                    registro["piscina"] = 1
                elif "quincho" in texto_fila and "sí" in texto_fila:
                    registro["quincho"] = 1
                elif "terraza" in texto_fila and "sí" in texto_fila:
                    registro["terraza"] = 1
                elif "gimnasio" in texto_fila and "sí" in texto_fila:
                    registro["gimnasio"] = 1
                elif "lavander" in texto_fila and "sí" in texto_fila: 
                    registro["lavanderia"] = 1

            datos_finales.append(registro)

        except Exception as e:
            print(f"Tabla no encontrada o página caída. Guardando con valores base.")
            datos_finales.append(registro)
            
        time.sleep(2) 

except Exception as e:
    print(f"Error crítico en Selenium: {e}")

finally:
    if driver is not None:
        driver.quit()
        print("\nNavegador cerrado.")

# ==============================================================================
# FASE 3: GUARDADO EN MONGODB (UPSERT)
# ==============================================================================
print("\n--- Proceso de guardado ---")
try:
    if datos_finales:
        client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
        db = client["Prueba"]
        coleccion = db["Pruebaaa"] 

        registros_nuevos = 0
        registros_actualizados = 0

        for d in datos_finales:
            resultado = coleccion.update_one(
                {"url": d["url"]}, 
                {"$set": d},       
                upsert=True        
            )

            if resultado.upserted_id:
                registros_nuevos += 1
            else:
                registros_actualizados += 1

        print(f"{datos_finales[0]['responsable']}")
        print(f"   -> {registros_nuevos} departamentos creados.")
        print(f"   -> {registros_actualizados} departamentos actualizados.")
    else:
        print("No se encontró ningún dato para guardar.")

except Exception as e:
    print(f"Error conectando a la Base de Datos: {e}")

🧹 Entorno virtual listo. Asignación: Páginas 1 a la 2.

--- [FASE 1] Recolectando enlaces de la Página 1 ---

--- [FASE 1] Recolectando enlaces de la Página 2 ---
Catálogo listo: 95 propiedades encontradas. Recolectando amenidades...
[1/95] Inspeccionando: Arriendo Departamento Amoblado...
[2/95] Inspeccionando: Departamento Avenida Del Mar I...
[3/95] Inspeccionando: Departamento Emilio Apey Id: 1...
[4/95] Inspeccionando: Departamento Avenida Costanera...
[5/95] Inspeccionando: Arriendo San Joaquin Barrio Un...
[6/95] Inspeccionando: Arriendo Departamento La Seren...
[7/95] Inspeccionando: Departamento Nuevo Orientación...
[8/95] Inspeccionando: Gran Departamento Amoblado En ...
[9/95] Inspeccionando: Arriendo Departamento La Seren...
[10/95] Inspeccionando: Arriendo Anual Departamento 2 ...
[11/95] Inspeccionando: Hermoso Departamento Frenta A ...
[12/95] Inspeccionando: Departamento En Arriendo Condo...
[13/95] Inspeccionando: Se Arrienda Departamento Amobl...
[14/95] Inspeccionand